In [31]:
%pip install geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 37.7 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [geopandas]/4 [geopandas]

[notice] A new release of pip is available: 25.2 -> 26.0
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [33]:
from datetime import date
import meteostat as ms
from meteostat import Point, daily
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
import sqlite3
import requests
from datetime import date
import meteostat as ms
import pandas as pd

In [34]:


# 1) Download the stations database (once)
STATIONS_DB_URL = "https://data.meteostat.net/stations.db"
STATIONS_DB_FILE = "stations.db"

with requests.get(STATIONS_DB_URL, stream=True) as r:
    r.raise_for_status()
    with open(STATIONS_DB_FILE, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

# 2) Open the DB and get all station IDs in France
conn = sqlite3.connect(STATIONS_DB_FILE)
cur = conn.cursor()

# country code 'FR' for France
cur.execute("SELECT id,region,latitude, longitude FROM stations WHERE country = 'FR'")
rows = cur.fetchall()
conn.close()

# Opret en Pandas DataFrame direkte fra resultatet for nem håndtering
# Kolonnerne vi skal bruge er 'id', 'latitude' og 'longitude'
df_stations = pd.DataFrame(rows, columns=['station_id', 'region','lat', 'lon'])

print(f"DataFrame created with {len(df_stations)} stations and coordinates.")
print(df_stations.head())


DataFrame created with 223 stations and coordinates.
  station_id region      lat     lon
0      07666      K  42.9183  3.0617
1      07690      U  43.6500  7.2000
2      07003      O  50.5167  1.6167
3      LFYL0      I  47.7000  6.5500
4      07222      R  47.1667 -1.6000


In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point


# 2. Produktionsdata (Eksempel for Eure-et-Loir '28', Seine-et-Marne '77')
prod_data = {'departement_code': ['28', '77', 'rest'], 
             'production_tonnes': [1500000, 1200000, 32300000]} # Total ~35M tons
df_production = pd.DataFrame(prod_data)

# 3. Load din GeoJSON fil (Sørg for det er DEPARTEMENTS.geojson)
gdf_departements = gpd.read_file("Meta/departements-1000m.geojson")

# --- STEP 1: Forbered Vejrstationer til Geospatial Analyse ---

# Omdan din stations DataFrame til en GeoDataFrame (kræver 'geometry' kolonne)
gdf_stations = gpd.GeoDataFrame(
    df_stations, 
    geometry=gpd.points_from_xy(df_stations.lon, df_stations.lat), 
    crs="EPSG:4326" # Standard koordinatsystem (WGS84)
)

# Sørg for at begge GeoDataFrames har samme koordinatsystem
gdf_departements = gdf_departements.to_crs("EPSG:4326")

# --- STEP 2: Geospatial Join ---
stations_with_dep = gpd.sjoin(gdf_stations, gdf_departements, how="inner", predicate="within")

# Da vi nu ved, at kolonnen hedder 'code', omdøber vi den her:
stations_with_dep = stations_with_dep.rename(columns={'code': 'departement_code'})

# --- STEP 3: Beregn Vægtene ---

# Flet med produktionsdata
stations_final = pd.merge(stations_with_dep, df_production, on='departement_code', how='left')

# Fjern stationer, der ikke har match i din produktions-tabel (f.eks. hvis de lander i et dep, du ikke har data for)
stations_final = stations_final.dropna(subset=['production_tonnes'])

# Beregn den samlede produktion for de departementer, der rent faktisk er dækket af stationer
# Vi tager summen af de unikke produktionsværdier for at undgå at tælle samme departement flere gange
total_prod_covered = stations_final.drop_duplicates('departement_code')['production_tonnes'].sum()

# Beregn vægten for hver station:
# Formel: (Dep_Produktion / Total_Produktion) / Antal_stationer_i_dette_dep
stations_final['weight'] = (
    stations_final['production_tonnes'] / 
    total_prod_covered / 
    stations_final.groupby('departement_code')['station_id'].transform('count')
)

# --- Tjek resultatet ---
print(f"Total produktion dækket: {total_prod_covered:,} tons")
print(stations_final[['station_id', 'departement_code', 'production_tonnes', 'weight']])




Total produktion dækket: 2,700,000.0 tons
   station_id departement_code  production_tonnes    weight
59      07140               28          1500000.0  0.277778
83      07153               77          1200000.0  0.444444
92      07143               28          1500000.0  0.277778

Sum af alle vægte (skal være 1.0): 1.0
